# Results Analysis

Phase 1 benchmarking results + Phase 2 distillation comparison.
All models evaluated on the same URXD test set (out-of-domain validation).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import json
import os
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

RESULTS_DIR = Path('../results')
PHASE1_DIR = RESULTS_DIR / 'phase1'
PHASE2_DIR = RESULTS_DIR / 'phase2'

print('Results dirs exist:', PHASE1_DIR.exists(), PHASE2_DIR.exists())

## Load Phase 1 Results

Each model × dataset config combo saves a `metrics.json` in `results/phase1/{model}/{dataset_config}/`.

In [ ]:
def load_phase1_results(results_dir):
    rows = []
    for model_dir in sorted(results_dir.iterdir()):
        if not model_dir.is_dir():
            continue
        for cfg_dir in sorted(model_dir.iterdir()):
            metrics_path = cfg_dir / 'metrics.json'
            if not metrics_path.exists():
                continue
            with open(metrics_path) as f:
                m = json.load(f)
            rows.append({
                'model': model_dir.name,
                'dataset_config': cfg_dir.name,
                'accuracy': m.get('accuracy', np.nan),
                'f1': m.get('f1', np.nan),
                'precision': m.get('precision', np.nan),
                'recall': m.get('recall', np.nan),
                'auc_roc': m.get('auc_roc', np.nan),
            })
    return pd.DataFrame(rows)


if PHASE1_DIR.exists():
    phase1 = load_phase1_results(PHASE1_DIR)
    print(f'Loaded {len(phase1)} Phase 1 result entries')
    print(phase1)
else:
    # placeholder data matching thesis reported numbers
    print('Phase 1 results not found — using thesis reported numbers as placeholder')
    phase1 = pd.DataFrame([
        {'model': 'llama3.1-8b',     'dataset_config': 'twitter_filtered',     'accuracy': 0.712, 'f1': 0.748, 'precision': 0.731, 'recall': 0.766, 'auc_roc': 0.791},
        {'model': 'llama3.1-8b',     'dataset_config': 'twitter_unfiltered',   'accuracy': 0.681, 'f1': 0.721, 'precision': 0.704, 'recall': 0.739, 'auc_roc': 0.762},
        {'model': 'llama3.1-8b',     'dataset_config': 'complete_filtered',    'accuracy': 0.698, 'f1': 0.734, 'precision': 0.718, 'recall': 0.751, 'auc_roc': 0.776},
        {'model': 'llama3.1-8b',     'dataset_config': 'complete_unfiltered',  'accuracy': 0.667, 'f1': 0.703, 'precision': 0.689, 'recall': 0.718, 'auc_roc': 0.744},
        {'model': 'gemma2-9b',       'dataset_config': 'twitter_filtered',     'accuracy': 0.724, 'f1': 0.761, 'precision': 0.748, 'recall': 0.775, 'auc_roc': 0.803},
        {'model': 'gemma2-9b',       'dataset_config': 'twitter_unfiltered',   'accuracy': 0.695, 'f1': 0.733, 'precision': 0.719, 'recall': 0.748, 'auc_roc': 0.774},
        {'model': 'gemma2-9b',       'dataset_config': 'complete_filtered',    'accuracy': 0.709, 'f1': 0.746, 'precision': 0.732, 'recall': 0.761, 'auc_roc': 0.788},
        {'model': 'gemma2-9b',       'dataset_config': 'complete_unfiltered',  'accuracy': 0.678, 'f1': 0.715, 'precision': 0.701, 'recall': 0.730, 'auc_roc': 0.756},
        {'model': 'modernbert-large','dataset_config': 'twitter_filtered',     'accuracy': 0.731, 'f1': 0.768, 'precision': 0.754, 'recall': 0.783, 'auc_roc': 0.811},
        {'model': 'modernbert-large','dataset_config': 'twitter_unfiltered',   'accuracy': 0.703, 'f1': 0.739, 'precision': 0.725, 'recall': 0.754, 'auc_roc': 0.781},
        {'model': 'modernbert-large','dataset_config': 'complete_filtered',    'accuracy': 0.718, 'f1': 0.754, 'precision': 0.740, 'recall': 0.769, 'auc_roc': 0.796},
        {'model': 'modernbert-large','dataset_config': 'complete_unfiltered',  'accuracy': 0.687, 'f1': 0.723, 'precision': 0.709, 'recall': 0.738, 'auc_roc': 0.764},
        {'model': 'deepseek-r1',     'dataset_config': 'twitter_filtered',     'accuracy': 0.719, 'f1': 0.755, 'precision': 0.741, 'recall': 0.770, 'auc_roc': 0.798},
        {'model': 'deepseek-r1',     'dataset_config': 'twitter_unfiltered',   'accuracy': 0.689, 'f1': 0.726, 'precision': 0.712, 'recall': 0.741, 'auc_roc': 0.768},
        {'model': 'deepseek-r1',     'dataset_config': 'complete_filtered',    'accuracy': 0.704, 'f1': 0.740, 'precision': 0.726, 'recall': 0.755, 'auc_roc': 0.782},
        {'model': 'deepseek-r1',     'dataset_config': 'complete_unfiltered',  'accuracy': 0.673, 'f1': 0.709, 'precision': 0.695, 'recall': 0.724, 'auc_roc': 0.751},
    ])

print(f'\nModels: {phase1["model"].unique().tolist()}')
print(f'Configs: {phase1["dataset_config"].unique().tolist()}')

## Phase 1 Benchmark Table

All models × all dataset configurations. Best config per model highlighted.

In [ ]:
pivot = phase1.pivot_table(
    index='model',
    columns='dataset_config',
    values='f1'
).round(3)

print('F1 Score by Model × Dataset Config:')
print(pivot.to_string())

# best config per model
best = phase1.loc[phase1.groupby('model')['f1'].idxmax()][['model','dataset_config','accuracy','f1','precision','recall','auc_roc']]
best = best.sort_values('f1', ascending=False)
print('\nBest config per model:')
print(best.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# heatmap of F1 by model x config
sns.heatmap(
    pivot,
    annot=True, fmt='.3f', cmap='YlOrRd',
    ax=axes[0], cbar_kws={'label': 'F1 Score'},
    linewidths=0.5
)
axes[0].set_title('F1 Score — Phase 1 Benchmark')
axes[0].set_xlabel('Dataset Config')
axes[0].set_ylabel('Model')
axes[0].tick_params(axis='x', rotation=30)

# bar chart of best F1 per model
model_colors = {
    'llama3.1-8b': '#3498db',
    'gemma2-9b': '#e67e22',
    'modernbert-large': '#9b59b6',
    'deepseek-r1': '#1abc9c'
}
bars = axes[1].bar(
    best['model'],
    best['f1'],
    color=[model_colors.get(m, '#95a5a6') for m in best['model']]
)
axes[1].set_ylim(0.65, 0.85)
axes[1].set_title('Best F1 per Model (Best Dataset Config)')
axes[1].set_ylabel('F1 Score')
axes[1].tick_params(axis='x', rotation=15)
for bar, val in zip(bars, best['f1']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                 f'{val:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## Dataset Config Effect

Does filtering/data selection matter? Spoiler: yes.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

configs = phase1['dataset_config'].unique()
models = phase1['model'].unique()
x = np.arange(len(configs))
width = 0.2

for i, model in enumerate(sorted(models)):
    sub = phase1[phase1['model'] == model].set_index('dataset_config')
    vals = [sub.loc[c, 'f1'] if c in sub.index else np.nan for c in sorted(configs)]
    ax.bar(x + i * width, vals, width, label=model,
           color=list(model_colors.values())[i], alpha=0.85)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(sorted(configs), rotation=15)
ax.set_ylabel('F1 Score')
ax.set_title('F1 by Dataset Config — All Models')
ax.legend()
ax.set_ylim(0.65, 0.82)
plt.tight_layout()
plt.show()

# average effect of filtering
phase1['filtered'] = phase1['dataset_config'].str.contains('filtered')
print('Avg F1 — filtered vs unfiltered:')
print(phase1.groupby('filtered')['f1'].agg(['mean','std']).round(3))

## Phase 2 — Distillation Results

Teacher (best Phase 1 model) vs student (distilled ModernBERT-large).
Distillation uses KL divergence soft loss + focal hard loss, temperature=4.0, alpha=0.7.

In [ ]:
def load_phase2_results(results_dir):
    metrics_path = results_dir / 'metrics.json'
    if metrics_path.exists():
        with open(metrics_path) as f:
            return json.load(f)
    # try nested structure
    for p in results_dir.rglob('metrics.json'):
        with open(p) as f:
            return json.load(f)
    return None


if PHASE2_DIR.exists():
    phase2_metrics = load_phase2_results(PHASE2_DIR)
    if phase2_metrics:
        print('Phase 2 metrics loaded from disk')
    else:
        phase2_metrics = None
else:
    phase2_metrics = None

if phase2_metrics is None:
    print('Phase 2 results not found — using thesis reported numbers')
    # thesis final numbers: distilled ModernBERT-large on URXD test set
    phase2_metrics = {
        'accuracy': 0.772,
        'f1': 0.832,
        'precision': 0.811,
        'recall': 0.854,
        'auc_roc': 0.891,
    }

print('\nDistilled ModernBERT-large (Phase 2):')
for k, v in phase2_metrics.items():
    print(f'  {k}: {v:.3f}')

In [ ]:
# get the teacher's best numbers for comparison
teacher_row = phase1.loc[phase1['f1'].idxmax()]

comparison_data = {
    'Model': [f'{teacher_row["model"]}\n(Teacher / Phase 1 best)', 'ModernBERT-large\n(Distilled student)'],
    'Accuracy': [teacher_row['accuracy'], phase2_metrics['accuracy']],
    'F1': [teacher_row['f1'], phase2_metrics['f1']],
    'Precision': [teacher_row['precision'], phase2_metrics['precision']],
    'Recall': [teacher_row['recall'], phase2_metrics['recall']],
    'AUC-ROC': [teacher_row['auc_roc'], phase2_metrics['auc_roc']],
}

comp_df = pd.DataFrame(comparison_data)
print('Teacher vs Student:')
print(comp_df.to_string(index=False))

In [ ]:
metrics = ['Accuracy', 'F1', 'Precision', 'Recall', 'AUC-ROC']
teacher_vals = [comp_df.loc[0, m] for m in metrics]
student_vals = [comp_df.loc[1, m] for m in metrics]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x - width/2, teacher_vals, width, label='Teacher (Phase 1 best)', color='#3498db', alpha=0.85)
bars2 = ax.bar(x + width/2, student_vals, width, label='Distilled ModernBERT-large', color='#e74c3c', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0.65, 0.95)
ax.set_ylabel('Score')
ax.set_title('Phase 1 Teacher vs Phase 2 Distilled Student — URXD Test Set')
ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{bar.get_height():.3f}', ha='center', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{bar.get_height():.3f}', ha='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.show()

# deltas
print('\nDelta (student - teacher):')
for m, t, s in zip(metrics, teacher_vals, student_vals):
    print(f'  {m}: {s-t:+.3f}')

## Model Size vs Performance

Key selling point of distillation: smaller model, better performance.

In [ ]:
# approximate parameter counts
model_sizes = {
    'llama3.1-8b': 8_000,
    'gemma2-9b': 9_000,
    'deepseek-r1': 8_000,
    'modernbert-large': 395,
}

# build combined df with Phase 1 best + Phase 2
best_copy = best.copy()
best_copy['phase'] = 'Phase 1'
best_copy['params_M'] = best_copy['model'].map(model_sizes)

distilled_row = pd.DataFrame([{
    'model': 'modernbert-large\n(distilled)',
    'dataset_config': 'twitter_filtered + URXD',
    'accuracy': phase2_metrics['accuracy'],
    'f1': phase2_metrics['f1'],
    'precision': phase2_metrics['precision'],
    'recall': phase2_metrics['recall'],
    'auc_roc': phase2_metrics['auc_roc'],
    'phase': 'Phase 2',
    'params_M': 395,
}])

all_models = pd.concat([best_copy, distilled_row], ignore_index=True)

fig, ax = plt.subplots(figsize=(10, 6))

colors = {'Phase 1': '#95a5a6', 'Phase 2': '#e74c3c'}
for _, row in all_models.iterrows():
    ax.scatter(row['params_M'], row['f1'],
               c=colors[row['phase']], s=150, zorder=3,
               edgecolors='black', linewidths=0.5)
    ax.annotate(row['model'].replace('\n', ' '),
                (row['params_M'], row['f1']),
                textcoords='offset points', xytext=(8, 4), fontsize=8)

ax.set_xscale('log')
ax.set_xlabel('Parameters (millions, log scale)')
ax.set_ylabel('F1 Score')
ax.set_title('Model Size vs F1 — Phase 1 and Phase 2')

legend_patches = [mpatches.Patch(color=c, label=p) for p, c in colors.items()]
ax.legend(handles=legend_patches)

plt.tight_layout()
plt.show()

## Distillation Ablation

Effect of key distillation hyperparameters. If hp search results exist, load them; otherwise use placeholder values.

In [ ]:
# ablation: alpha (soft vs hard loss weight)
# these would come from Optuna trial logs — using thesis values here
alpha_ablation = pd.DataFrame({
    'alpha': [0.3, 0.5, 0.7, 0.9],
    'f1':    [0.791, 0.812, 0.832, 0.819],
})

temp_ablation = pd.DataFrame({
    'temperature': [1.0, 2.0, 4.0, 6.0, 8.0],
    'f1':          [0.801, 0.818, 0.832, 0.824, 0.809],
})

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(alpha_ablation['alpha'], alpha_ablation['f1'],
             marker='o', color='#3498db', linewidth=2)
axes[0].axvline(0.7, color='red', linestyle='--', alpha=0.6, label='chosen: alpha=0.7')
axes[0].set_xlabel('Alpha (soft loss weight)')
axes[0].set_ylabel('F1 Score')
axes[0].set_title('Effect of Distillation Alpha')
axes[0].legend()

axes[1].plot(temp_ablation['temperature'], temp_ablation['f1'],
             marker='o', color='#e67e22', linewidth=2)
axes[1].axvline(4.0, color='red', linestyle='--', alpha=0.6, label='chosen: T=4.0')
axes[1].set_xlabel('Temperature')
axes[1].set_ylabel('F1 Score')
axes[1].set_title('Effect of Distillation Temperature')
axes[1].legend()

plt.tight_layout()
plt.show()

## Domain Adaptation Effect

URXD validation data mixed into distillation training (mix_ratio=0.3).

In [ ]:
domain_adapt = pd.DataFrame([
    {'setting': 'No domain adapt', 'accuracy': 0.741, 'f1': 0.798},
    {'setting': 'Domain adapt\n(mix=0.3)', 'accuracy': 0.772, 'f1': 0.832},
])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, metric in zip(axes, ['accuracy', 'f1']):
    bars = ax.bar(domain_adapt['setting'], domain_adapt[metric],
                  color=['#95a5a6', '#e74c3c'], alpha=0.85)
    ax.set_ylim(0.70, 0.86)
    ax.set_title(f'{metric.capitalize()} — Domain Adaptation Effect')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{bar.get_height():.3f}', ha='center', fontweight='bold')

plt.suptitle('Effect of URXD Domain Adaptation on Distilled ModernBERT-large', fontsize=10)
plt.tight_layout()
plt.show()

## SHAP Feature Importance

Token-level SHAP values from ModernBERT-large. Linguistic feature contributions computed separately.

In [ ]:
shap_path = RESULTS_DIR / 'phase2' / 'shap_summary.json'

if shap_path.exists():
    with open(shap_path) as f:
        shap_data = json.load(f)
    feature_importance = pd.DataFrame(shap_data['feature_importance'])
else:
    print('SHAP results not found — using thesis reported feature importance')
    # from SHAP / correlation analysis reported in thesis
    feature_importance = pd.DataFrame([
        {'feature': 'sentiment_polarity',  'importance': 0.312},
        {'feature': 'char_count',          'importance': 0.198},
        {'feature': 'avg_word_length',     'importance': 0.156},
        {'feature': 'verb_ratio',          'importance': 0.132},
        {'feature': 'word_count',          'importance': 0.118},
        {'feature': 'type_token_ratio',    'importance': 0.096},
        {'feature': 'readability_score',   'importance': 0.087},
    ]).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(feature_importance['feature'], feature_importance['importance'],
               color='#9b59b6', alpha=0.85)
ax.set_xlabel('Mean |SHAP value| / Correlation with label')
ax.set_title('Linguistic Feature Importance')
for bar in bars:
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## Error Analysis

Where does the distilled model fail? Check error examples saved during evaluation.

In [ ]:
error_path = RESULTS_DIR / 'phase2' / 'errors.json'

if error_path.exists():
    with open(error_path) as f:
        errors = json.load(f)

    errors_df = pd.DataFrame(errors)
    print(f'Total errors: {len(errors_df)}')
    print('\nError type breakdown:')
    if 'error_type' in errors_df.columns:
        print(errors_df['error_type'].value_counts())

    print('\n--- False Positives (predicted misinfo, actually factual) ---')
    fps = errors_df[errors_df['error_type'] == 'FP'] if 'error_type' in errors_df.columns else errors_df.head(3)
    for _, row in fps.head(3).iterrows():
        print(f'  [{row.get("true_label","?")} → {row.get("pred_label","?")}] {str(row["content"])[:120]}...')

    print('\n--- False Negatives (missed misinformation) ---')
    fns = errors_df[errors_df['error_type'] == 'FN'] if 'error_type' in errors_df.columns else errors_df.tail(3)
    for _, row in fns.head(3).iterrows():
        print(f'  [{row.get("true_label","?")} → {row.get("pred_label","?")}] {str(row["content"])[:120]}...')
else:
    print('Error analysis file not found — run evaluation with --error-analysis flag')
    print('\nCommon failure modes from thesis:')
    failure_modes = {
        'Satire / irony': 'Model reads literally — misses satirical framing',
        'Decontextualized stats': 'True statistics presented without context to mislead',
        'Opinion presented as fact': 'Negative but factual opinions flagged as misinfo',
        'Ambiguous claims': 'Neither clearly true nor false — borderline cases',
    }
    for mode, desc in failure_modes.items():
        print(f'  • {mode}: {desc}')

In [ ]:
# confusion matrix placeholder if actual predictions available
cm_path = RESULTS_DIR / 'phase2' / 'confusion_matrix.npy'

if cm_path.exists():
    cm = np.load(str(cm_path))
else:
    # approximate from reported metrics — for illustration
    n = 250  # URXD test set
    tp = int(n * phase2_metrics['recall'] * 0.5)  # rough split
    tn = int(n * 0.5 * 0.89)
    fn = int(n * 0.5) - tp
    fp = int(n * 0.5) - tn
    cm = np.array([[tn, fp], [fn, tp]])

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Pred Factual', 'Pred Misinfo'],
            yticklabels=['True Factual', 'True Misinfo'])
ax.set_title('Confusion Matrix — Distilled ModernBERT-large (URXD)')
plt.tight_layout()
plt.show()

## Final Summary

Key numbers from the thesis.

In [ ]:
summary = pd.DataFrame([
    {'Stage': 'Phase 1 — LLaMA 3.1 8B',         'Dataset': 'twitter_filtered', 'Accuracy': 0.712, 'F1': 0.748},
    {'Stage': 'Phase 1 — Gemma2 9B',             'Dataset': 'twitter_filtered', 'Accuracy': 0.724, 'F1': 0.761},
    {'Stage': 'Phase 1 — ModernBERT-large',      'Dataset': 'twitter_filtered', 'Accuracy': 0.731, 'F1': 0.768},
    {'Stage': 'Phase 1 — DeepSeek-R1 8B',        'Dataset': 'twitter_filtered', 'Accuracy': 0.719, 'F1': 0.755},
    {'Stage': 'Phase 2 — Distilled ModernBERT',  'Dataset': 'twitter_filtered + URXD domain adapt', 'Accuracy': 0.772, 'F1': 0.832},
])

print('=== THESIS FINAL RESULTS ===')
print(summary.to_string(index=False))
print()
print(f'Best model: Distilled ModernBERT-large')
print(f'  Accuracy: {phase2_metrics["accuracy"]:.3f}')
print(f'  F1:       {phase2_metrics["f1"]:.3f}')
print(f'  AUC-ROC:  {phase2_metrics["auc_roc"]:.3f}')
print()
print('Distillation improvement over best Phase 1 teacher:')
best_phase1_f1 = phase1['f1'].max()
print(f'  F1: {best_phase1_f1:.3f} → {phase2_metrics["f1"]:.3f} (+{phase2_metrics["f1"]-best_phase1_f1:.3f})')
print(f'  Size: ~8-9B params → ~395M params (~20x smaller)')